In [2]:
# ── Cell 1: Environment Setup ─────────────────────────────────────
# Run this FIRST every time before anything else
import os
import sys

# Point PySpark to Python 3.11 (required — Python 3.14 is incompatible)
os.environ["PYSPARK_PYTHON"]        = r"C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\python.exe"

# Hadoop winutils path (required on Windows for Spark)
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = os.environ["PATH"] + r";C:\hadoop\bin"

print(f"✓ Python (driver):  {sys.version}")
print(f"✓ HADOOP_HOME:      {os.environ['HADOOP_HOME']}")
print(f"✓ PYSPARK_PYTHON:   {os.environ['PYSPARK_PYTHON']}")
print("Environment ready ✓")

✓ Python (driver):  3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
✓ HADOOP_HOME:      C:\hadoop
✓ PYSPARK_PYTHON:   C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\python.exe
Environment ready ✓


In [3]:
# ── Cell 2: Start Spark Session ───────────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01-Lakehouse-Foundation") \
    .master("local[*]") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .getOrCreate()

# Suppress verbose logs — only show errors
spark.sparkContext.setLogLevel("ERROR")

print(f"✓ Spark version:  {spark.version}")
print(f"✓ Python version: {spark.sparkContext.pythonVer}")
print("Spark session ready ✓")

C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


✓ Spark version:  4.2.0
✓ Python version: 3.11
Spark session ready ✓


In [5]:
# ── Cell 3: Create Sample Dataset ────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Retail sales data — relevant to your background
data = [
    ("2026-01-05", "London",     "Electronics", "Laptop",  1, 899.99),
    ("2026-01-05", "Manchester", "Clothing",    "Jacket",  3,  49.99),
    ("2026-01-06", "London",     "Electronics", "Phone",   2, 599.99),
    ("2026-01-06", "Birmingham", "Food",        "Coffee", 10,   4.99),
    ("2026-01-07", "Manchester", "Electronics", "Tablet",  1, 349.99),
    ("2026-01-07", "London",     "Clothing",    "Shoes",   2,  79.99),
    ("2026-01-08", "Birmingham", "Electronics", "Laptop",  1, 899.99),
    ("2026-01-08", "London",     "Food",        "Tea",    20,   2.99),
    ("2026-01-09", "Manchester", "Clothing",    "Jacket",  1,  49.99),
    ("2026-01-09", "Birmingham", "Electronics", "Phone",   1, 599.99),
]

schema = StructType([
    StructField("sale_date",  StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("product",    StringType(),  True),
    StructField("quantity",   IntegerType(), True),
    StructField("unit_price", DoubleType(),  True),
])

df_raw = spark.createDataFrame(data, schema)

print(f"✓ Rows loaded: {df_raw.count()}")
df_raw.printSchema()
df_raw.show()

✓ Rows loaded: 10
root
 |-- sale_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+----------+----------+-----------+-------+--------+----------+
| sale_date|    region|   category|product|quantity|unit_price|
+----------+----------+-----------+-------+--------+----------+
|2026-01-05|    London|Electronics| Laptop|       1|    899.99|
|2026-01-05|Manchester|   Clothing| Jacket|       3|     49.99|
|2026-01-06|    London|Electronics|  Phone|       2|    599.99|
|2026-01-06|Birmingham|       Food| Coffee|      10|      4.99|
|2026-01-07|Manchester|Electronics| Tablet|       1|    349.99|
|2026-01-07|    London|   Clothing|  Shoes|       2|     79.99|
|2026-01-08|Birmingham|Electronics| Laptop|       1|    899.99|
|2026-01-08|    London|       Food|    Tea|      20|      2.99|
|2026-01-09|Manchester|   Cloth

In [6]:
# ── Cell 4: Transform Data ────────────────────────────────────────
df_clean = (
    df_raw
    # Cast date string to proper date type
    .withColumn("sale_date",     F.to_date(F.col("sale_date"), "yyyy-MM-dd"))
    # Calculate total revenue per line
    .withColumn("total_revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
    # Extract year and month for partitioning
    .withColumn("year",          F.year(F.col("sale_date")))
    .withColumn("month",         F.month(F.col("sale_date")))
    # Standardise region to uppercase
    .withColumn("region",        F.upper(F.col("region")))
    # Add audit timestamp
    .withColumn("processed_at",  F.current_timestamp())
)

print("✓ Transformation complete")
print(f"✓ Columns: {df_clean.columns}")
print()
df_clean.printSchema()
df_clean.show(truncate=False)

✓ Transformation complete
✓ Columns: ['sale_date', 'region', 'category', 'product', 'quantity', 'unit_price', 'total_revenue', 'year', 'month', 'processed_at']

root
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- processed_at: timestamp (nullable = false)

+----------+----------+-----------+-------+--------+----------+-------------+----+-----+--------------------------+
|sale_date |region    |category   |product|quantity|unit_price|total_revenue|year|month|processed_at              |
+----------+----------+-----------+-------+--------+----------+-------------+----+-----+--------------------------+
|2026-01-05|LONDON    |Electronics|Laptop |1       |899.99    |899.99       |2

In [7]:
# ── Cell 5: Analyse with Spark SQL ───────────────────────────────
# Register as temp view — same concept as a Fabric Lakehouse table
df_clean.createOrReplaceTempView("sales_clean")
print("✓ Registered temp view: sales_clean")
print()

# Query 1 — Revenue by region
print("=== Revenue by Region ===")
spark.sql("""
    SELECT region,
           ROUND(SUM(total_revenue), 2) AS total_revenue,
           SUM(quantity)                AS total_units,
           COUNT(DISTINCT product)      AS distinct_products
    FROM sales_clean
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

# Query 2 — Revenue by category
print("=== Revenue by Category ===")
spark.sql("""
    SELECT category,
           ROUND(SUM(total_revenue), 2) AS total_revenue
    FROM sales_clean
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

# Query 3 — Top products
print("=== Top Products ===")
spark.sql("""
    SELECT product,
           ROUND(SUM(total_revenue), 2) AS total_revenue,
           SUM(quantity)                AS units_sold
    FROM sales_clean
    GROUP BY product
    ORDER BY total_revenue DESC
""").show()

✓ Registered temp view: sales_clean

=== Revenue by Region ===
+----------+-------------+-----------+-----------------+
|    region|total_revenue|total_units|distinct_products|
+----------+-------------+-----------+-----------------+
|    LONDON|      2319.75|         25|                4|
|BIRMINGHAM|      1549.88|         12|                3|
|MANCHESTER|       549.95|          5|                2|
+----------+-------------+-----------+-----------------+

=== Revenue by Category ===
+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|      3949.94|
|   Clothing|       359.94|
|       Food|        109.7|
+-----------+-------------+

=== Top Products ===
+-------+-------------+----------+
|product|total_revenue|units_sold|
+-------+-------------+----------+
| Laptop|      1799.98|         2|
|  Phone|      1799.97|         3|
| Tablet|       349.99|         1|
| Jacket|       199.96|         4|
|  Shoes|       159.98|         2|
|    Tea|  

In [10]:
# ── Cell 6: Save Data & Summary ───────────────────────────────────
import os

# Save to CSV using pandas (bypasses Windows/Hadoop limitations)
# In Fabric this step is replaced by .write.format("delta").saveAsTable()
output_dir  = os.path.join(os.path.expanduser("~"), "dp700-portfolio", "data")
output_file = os.path.join(output_dir, "sales_clean.csv")
os.makedirs(output_dir, exist_ok=True)

df_pandas = df_clean.toPandas()
df_pandas.to_csv(output_file, index=False)

print(f"✓ Rows saved:  {len(df_pandas)}")
print(f"✓ File path:   {output_file}")
print(f"✓ Columns:     {list(df_pandas.columns)}")
print()

# Final summary
print("=" * 50)
print("  NOTEBOOK COMPLETE ✓")
print("=" * 50)
print()
print("What this notebook demonstrated:")
print("  1. Create a Spark session (local mode)")
print("  2. Define schema and load raw data")
print("  3. Transform: cast types, derive columns, standardise")
print("  4. Analyse: aggregations with Spark SQL")
print("  5. Save output for downstream use")
print()
print("In Microsoft Fabric, steps 1-4 are identical.")
print("Step 5 becomes: .write.format('delta').saveAsTable()")
print("  — no winutils needed, OneLake handles storage.")

C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


✓ Rows saved:  10
✓ File path:   C:\Users\cazac\dp700-portfolio\data\sales_clean.csv
✓ Columns:     ['sale_date', 'region', 'category', 'product', 'quantity', 'unit_price', 'total_revenue', 'year', 'month', 'processed_at']

  NOTEBOOK COMPLETE ✓

What this notebook demonstrated:
  1. Create a Spark session (local mode)
  2. Define schema and load raw data
  3. Transform: cast types, derive columns, standardise
  4. Analyse: aggregations with Spark SQL
  5. Save output for downstream use

In Microsoft Fabric, steps 1-4 are identical.
Step 5 becomes: .write.format('delta').saveAsTable()
  — no winutils needed, OneLake handles storage.


C:\Users\cazac\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
